# Train **My Own AI Model** on Google Colab

Trains the from-scratch GPT (`llm-from-scratch/`) to completion on Colab's free compute — which runs a cell uninterrupted, unlike the ephemeral dev container. Produces a `web/model.json` you download and hand back to deploy to the live app at `/llm/`.

**How to use:** `Runtime → Run all`, then wait for the training cell to finish. **Best on a laptop** — a phone tab often disconnects mid-training, which leaves the model undertrained (gibberish). The training cell prints a sample at the end: only download if it reads like real sentences.

⚠️ This is a small model: it learns the *style* of the training text, not real facts. Names/dates it invents.

## 1. Get the code + dependencies

In [ ]:
!git clone --depth 1 https://github.com/Refayethossain28/BallrzAPP.git
%cd BallrzAPP/llm-from-scratch
!pip -q install numpy

## 2. Pick what to train on

- **`'hub'`** — *everything in the Ballrz Hub*: every app's name, description and write-up. The model ends up "dreaming" in the voice of your own apps. Small corpus → char-level + a compact model.
- **`'wikipedia'`** — WikiText-2 (~10 MB of real Wikipedia article text), English-only. Bigger model, encyclopedic voice.
- Or paste any plain-text **URL**.

This cell also picks sensible model settings for the choice.

In [ ]:
CORPUS = 'hub'   # 'hub' | 'wikipedia' | 'https://…/some.txt'

import urllib.request, re, os, shutil
os.makedirs('data', exist_ok=True)

if CORPUS == 'hub':
    !python build_hub_corpus.py
    shutil.copy('data/hub.txt', 'data/corpus.txt')
    TOK, VOCAB, NLAYER, NEMBD, BLOCK, STEPS = 'char', 96, 4, 128, 96, 2000
else:
    URL = ('https://raw.githubusercontent.com/pytorch/examples/main/word_language_model/data/wikitext-2/train.txt'
           if CORPUS == 'wikipedia' else CORPUS)
    urllib.request.urlretrieve(URL, 'data/corpus.txt')
    text = open('data/corpus.txt', encoding='utf-8', errors='ignore').read()
    text = text.replace('@,@', '').replace(' @.@ ', '.').replace(' @-@ ', '-').replace('<unk>', '')
    text = re.sub(r' ([,.;:!?)])', r'\1', text).replace('( ', '(')
    text = ''.join(ch for ch in text if ch == '\n' or 32 <= ord(ch) < 127)  # English/ASCII only
    text = re.sub(r'[ \t]{2,}', ' ', text)
    open('data/corpus.txt', 'w', encoding='utf-8').write(text)
    TOK, VOCAB, NLAYER, NEMBD, BLOCK, STEPS = 'bpe', 512, 6, 160, 96, 3000

print('corpus:', os.path.getsize('data/corpus.txt'), 'bytes | settings:', TOK, VOCAB, NLAYER, NEMBD, BLOCK, STEPS)

## 3. Train

Resumable: if the Colab runtime drops, just run this cell again and it continues from the last checkpoint. Wait for it to print the final step and a text sample.

In [ ]:
!python train.py --data data/corpus.txt --tokenizer {TOK} --vocab_size {VOCAB} \
    --n_layer {NLAYER} --n_head 8 --n_embd {NEMBD} --block_size {BLOCK} --batch_size 32 \
    --steps {STEPS} --lr 3e-4 --min_lr_ratio 0.05 --eval_every 200 --out ckpt.npz

## 4. Export the browser weights + preview a sample

In [ ]:
!python export_web.py --ckpt ckpt.npz --out web/model.json
!python sample.py --ckpt ckpt.npz --prompt 'The ' --tokens 200 --temperature 0.7 --top_p 0.9 --repetition_penalty 1.3

## 5. Download `model.json`

Check the sample above reads like real sentences first. Then send this file back to deploy it to the live app.

In [ ]:
from google.colab import files
print('model.json size:', os.path.getsize('web/model.json'), 'bytes')
files.download('web/model.json')